# Exercises: Hypothesis Testing

**A Waiter's Tips**

The following description was retrieved from Kaggle page.

> Food servers’ tips in restaurants may be influenced by many factors, including the nature of the restaurant, size of the party, and table locations in the restaurant. Restaurant managers need to know which factors matter when they assign tables to food servers. For the sake of staff morale, they usually want to avoid either the substance or the appearance of unfair treatment of the servers, for whom tips (at least in restaurants in the United States) are a major component of pay. In one restaurant, a food server recorded the following data on all customers they served during an interval of two and a half months in early 1990. The restaurant, located in a suburban shopping mall, was part of a national chain and served a varied menu. In observance of local law, the restaurant offered to seat in a non-smoking section to patrons who requested it. Each record includes a day and time, and taken together, they show the server’s work schedule.

Acknowledgements The data was reported in a collection of case studies for business statistics. Bryant, P. G. and Smith, M (1995) Practical Data Analysis: Case Studies in Business Statistics. Homewood, IL: Richard D. Irwin Publishing

The dataset is also available through the Python package Seaborn.

In [1]:
import seaborn as sns
tips = sns.load_dataset("tips")
tips.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


Here's a description of each column in the dataset:

- `total_bill`: The total bill amount, including the cost of food and drinks.
- `tip`: The tip amount given by the customer.
- `sex`: The gender of the customer (e.g., Male or Female).
- `smoker`: Whether the customer is a smoker or not (e.g., Yes or No).
- `day`: The day of the week when the transaction occurred (e.g., Sun, Sat, Thu, etc.).
- `time`: The time of day when the transaction occurred, typically categorized as Lunch or Dinner.
- `size`: The size of the party or group of customers.

**Your Task**: is to accept or reject the following hypothesis using statistical testing:

- Hypothesis $H_1$: smoking is associated with time of visit
- Hypothesis $H_2$: the bigger the group the higher the tip
- Hypothesis $H_3$: group size is different based on the time of visit
- Hypothesis $H_4$: (... come up with a hypothesis of your own ...)
- Finally, analyze if size (party size) is a **confounder**. That is, does a larger party cause a higher tip, or simply a higher bill which then leads to a higher tip?

---

In [3]:
# H1: smoking vs time

table = pd.crosstab(tips["smoker"], tips["time"])
_, p1, _, _ = stats.chi2_contingency(table)

print("p-value:", p1)

# Answer:
# There is no significant association between smoking and time of visit (p > 0.05).


p-value: 0.4771485672079724


In [4]:
# H2: group size vs tip

r, p2 = stats.pearsonr(tips["size"], tips["tip"])

print("p-value:", p2)

# Answer:
# There is a significant relationship between group size and tip (p < 0.05).

p-value: 4.3005433272249666e-16


In [5]:
# H3: size difference (lunch vs dinner)

lunch = tips[tips["time"] == "Lunch"]["size"]
dinner = tips[tips["time"] == "Dinner"]["size"]

_, p3 = stats.mannwhitneyu(lunch, dinner)

print("p-value:", p3)

# Answer:
# There is a significant difference in group size between lunch and dinner (p < 0.05).

p-value: 0.010166655373207559


In [8]:
tips["tip_pct"] = tips["tip"] / tips["total_bill"]

In [9]:
# H4: tip percentage (smoker vs non-smoker)

smoker = tips[tips["smoker"] == "Yes"]["tip_pct"]
non_smoker = tips[tips["smoker"] == "No"]["tip_pct"]

_, p4 = stats.mannwhitneyu(smoker, non_smoker)

print("p-value:", p4)

# Answer:
# There is no significant difference in tip percentage between smokers and non-smokers (p > 0.05).

p-value: 0.5607355643803174


In [10]:
# Confounder: size + total_bill

model = smf.ols("tip ~ size + total_bill", data=tips).fit()
print(model.summary())

# Answer:
# Both size and total_bill affect tip, but total_bill is the stronger predictor.
# This suggests that the effect of size is partly explained by larger bills.

                            OLS Regression Results                            
Dep. Variable:                    tip   R-squared:                       0.468
Model:                            OLS   Adj. R-squared:                  0.463
Method:                 Least Squares   F-statistic:                     105.9
Date:                Fri, 10 Apr 2026   Prob (F-statistic):           9.67e-34
Time:                        12:05:19   Log-Likelihood:                -347.99
No. Observations:                 244   AIC:                             702.0
Df Residuals:                     241   BIC:                             712.5
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.6689      0.194      3.455      0.0